In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

In [ ]:
gen_sam_all_0223 = pd.read_csv("../data/generated/sam_gen_0223.csv")
gen_sam_all_2024 = pd.read_csv("../data/generated/gen_sam_all.csv")
gen_sam_all = pd.concat([gen_sam_all_2024, gen_sam_all_0223])
gen_sam_all_drop = gen_sam_all.drop_duplicates(keep='first', subset='smiles').reset_index(drop=True)
print(gen_sam_all_drop.shape)

In [ ]:
gen_sam_all_drop.reset_index(drop=True, inplace=True)
print(gen_sam_all_drop.shape)

Substructures matching

In [ ]:
#按骨架核心分类
df_copy = gen_sam_all_drop.copy()

df_1 = pd.DataFrame(columns=df_copy.columns)
df_2 = pd.DataFrame(columns=df_copy.columns)
df_3 = pd.DataFrame(columns=df_copy.columns)
df_4 = pd.DataFrame(columns=df_copy.columns)
df_5 = pd.DataFrame(columns=df_copy.columns)
df_6 = pd.DataFrame(columns=df_copy.columns)
df_7 = pd.DataFrame(columns=df_copy.columns)

# 链状
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        ring_info = mol.GetRingInfo()
        num_rings = ring_info.NumRings()
        if num_rings == 0:
            df_1 = pd.concat([df_1, pd.DataFrame([row])], ignore_index=True)
            df_copy.drop(index, inplace=True)

# 萘酰胺
patt_1 = Chem.MolFromSmiles('O=C1NC(=O)C2=CC=CC3=C2C1=CC=C3')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
       print(f"Invalid SMILES: {smiles}")
       continue  # 跳过这一行
    if mol.HasSubstructMatch(patt_1):
        df_2 = pd.concat([df_2, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 三苯胺
patt_2 = Chem.MolFromSmiles('C1=CC=C(C=C1)N(C1=CC=CC=C1)C1=CC=CC=C1')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
       print(f"Invalid SMILES: {smiles}")
       continue  # 跳过这一行
    if mol.HasSubstructMatch(patt_2):
        df_3 = pd.concat([df_3, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 咔唑
patt_3 = Chem.MolFromSmiles('N1C2=C(C=CC=C2)C2=C1C=CC=C2')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
       print(f"Invalid SMILES: {smiles}")
       continue  # 跳过这一行
    if mol.HasSubstructMatch(patt_3):
        df_4 = pd.concat([df_4, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 联噻吩
patt_4 = Chem.MolFromSmiles('S1C=CC=C1C1=CC=CS1')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
       print(f"Invalid SMILES: {smiles}")
       continue  # 跳过这一行
    if mol.HasSubstructMatch(patt_4):
        df_5 = pd.concat([df_5, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 单环
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
       print(f"Invalid SMILES: {smiles}")
       continue  # 跳过这一行
    if mol is not None:
        ring_info = mol.GetRingInfo()
        num_rings = ring_info.NumRings()
        if num_rings == 1:
            df_6 = pd.concat([df_6, pd.DataFrame([row])], ignore_index=True)
            df_copy.drop(index, inplace=True)

# 其他
df_7 = df_copy.copy()

In [ ]:
print(df_1.shape)
print(df_2.shape)
print(df_3.shape)
print(df_4.shape)
print(df_5.shape)
print(df_6.shape)
print(df_7.shape)

In [ ]:
sam_substructures_matching = pd.concat([df_2, df_3, df_4, df_5, df_6])
print(sam_substructures_matching.shape)

In [ ]:
smiles_substructures_matching = sam_substructures_matching.loc[:, ["smiles"]].reset_index(drop=True)
smiles_substructures_matching

In [ ]:
sam_structure_matching = smiles_substructures_matching.iloc[smiles_substructures_matching.index[~smiles_substructures_matching.loc[:,"smiles"].isna()],:]
sam_structure_matching

In [ ]:
sam_structure_matching = sam_structure_matching.rename(columns={"smiles": "SMILES"})
sam_structure_matching 

In [ ]:
sam_structure_matching.to_csv("./sam_structure_matching_0225.csv")

Energy level matching

In [ ]:
sam_structure_matching = pd.read_csv("./sam_structure_matching_0225.csv")

In [ ]:
from unimol_tools import MolPredict
import os

In [ ]:
!pwd

In [ ]:
homo_weights_path = r"/data/Jn/Unimol/weights_fix_0226/homo/bs_32_lr_1e-4"
input_csv = r"./tmp.csv"

In [ ]:
pd.DataFrame(sam_structure_matching.iloc[:,1]).to_csv(input_csv, index=False)

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
homo_clf = MolPredict(load_model=homo_weights_path)

In [ ]:
# 基于SMILES的文件输入模式的预测 
homo_pred = homo_clf.predict(data=input_csv)

In [ ]:
homo_pred_results = pd.DataFrame({
    "smiles": homo_clf.datahub.data["smiles"],
    "homo_pred": homo_pred.flatten(),
    })
homo_pred_results.head(10)

In [ ]:
lumo_weights_path = r"/data/Jn/Unimol/weights_fix_0226/lumo/bs_32_lr_1e-4"

In [ ]:
lumo_clf = MolPredict(load_model=lumo_weights_path)

In [ ]:
lumo_pred = lumo_clf.predict(data=input_csv)

In [ ]:
homo_lumo_pred_results = pd.DataFrame({
    "smiles": homo_clf.datahub.data["smiles"],
    "homo_pred": homo_pred.flatten(),
    "lumo_pred": lumo_pred.flatten()
    })
homo_lumo_pred_results.head(10)

In [ ]:
homo_lumo_pred_results.shape

In [ ]:
!pwd

In [ ]:
homo_lumo_pred_results.to_csv("./homo_lumo_pred_results_0225.csv")

In [ ]:
os.remove(input_csv)

In [ ]:
homo_matching = homo_lumo_pred_results[(homo_lumo_pred_results['homo_pred'] >= -5.16) & (homo_lumo_pred_results['homo_pred'] <= -4.96)]

In [ ]:
homo_lumo_matching = homo_matching[(homo_matching['lumo_pred']>= -3.43)]
homo_lumo_matching

In [ ]:
homo_lumo_matching = homo_lumo_matching.rename(columns={"smiles": "SMILES"})

Dipole moment screening

In [ ]:
dm_input_csv = r"./dm_tmp.csv"

In [ ]:
pd.DataFrame(homo_lumo_matching.iloc[:,0]).to_csv(dm_input_csv, index=False)

In [ ]:
dm_weights_path = r"/data/Jn/Unimol/weights_fix_0226/dm/bs_32_lr_1e-4"

In [ ]:
dm_clf = MolPredict(load_model=dm_weights_path)

In [ ]:
dm_pred = dm_clf.predict(data=dm_input_csv)

In [ ]:
dm_homo_lumo_pred_results = pd.DataFrame({
    "smiles": homo_lumo_matching["SMILES"],
    "homo_pred": homo_lumo_matching["homo_pred"],
    "lumo_pred": homo_lumo_matching["lumo_pred"],
    "dm_pred": dm_pred.flatten(),
    })
dm_homo_lumo_pred_results.head(10)

In [ ]:
max(dm_homo_lumo_pred_results["dm_pred"])

In [ ]:
min(dm_homo_lumo_pred_results["dm_pred"])

In [ ]:
dm_homo_lumo_pred_results.to_csv("./dm_homo_lumo_pred_results_0225.csv")

In [ ]:
dm_homo_lumo_pred_results = pd.read_csv("./dm_homo_lumo_pred_results_0225.csv")

In [ ]:
sorted_dm_results = dm_homo_lumo_pred_results.sort_values(by="dm_pred", ascending=False)
sorted_dm_results

In [ ]:
top_20_dm_index = int(len(sorted_dm_results) * 0.20)

In [ ]:
dm_screening = sorted_dm_results.iloc[:top_20_dm_index]

In [ ]:
dm_screening

Efficiency prediction

In [ ]:
import pandas as pd
import warnings
import numpy as np
from sklearn.svm import SVC
from rdkit import Chem
from tqdm import tqdm
import warnings
from data_processing import dataset_process, calculate_ecfp, normalize_predict_sample
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
dataset_path = "./SAM_data_PCE_15%_0721.csv"

In [ ]:
dataset = pd.read_csv(dataset_path)

In [ ]:
# 调用dataset_process函数进行训练数据处理
features_to_include = ['ecfp', 'pvk_ions', 'pvk_bandgap', 'substrate']
processed_data, scaler_dict = dataset_process(dataset, features_to_include)

In [ ]:
processed_data

In [ ]:
#按骨架核心分类
df_copy = processed_data.copy()

df_1 = pd.DataFrame(columns=df_copy.columns)
df_2 = pd.DataFrame(columns=df_copy.columns)
df_3 = pd.DataFrame(columns=df_copy.columns)
df_4 = pd.DataFrame(columns=df_copy.columns)
df_5 = pd.DataFrame(columns=df_copy.columns)
df_6 = pd.DataFrame(columns=df_copy.columns)
df_7 = pd.DataFrame(columns=df_copy.columns)

# 链状
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        ring_info = mol.GetRingInfo()
        num_rings = ring_info.NumRings()
        if num_rings == 0:
            df_1 = pd.concat([df_1, pd.DataFrame([row])], ignore_index=True)
            df_copy.drop(index, inplace=True)

# 萘酰胺
patt_1 = Chem.MolFromSmiles('O=C1NC(=O)C2=CC=CC3=C2C1=CC=C3')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_1):
        df_2 = pd.concat([df_2, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 三苯胺
patt_2 = Chem.MolFromSmiles('C1=CC=C(C=C1)N(C1=CC=CC=C1)C1=CC=CC=C1')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_2):
        df_3 = pd.concat([df_3, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 咔唑
patt_3 = Chem.MolFromSmiles('N1C2=C(C=CC=C2)C2=C1C=CC=C2')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_3):
        df_4 = pd.concat([df_4, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 联噻吩
patt_4 = Chem.MolFromSmiles('S1C=CC=C1C1=CC=CS1')
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol.HasSubstructMatch(patt_4):
        df_5 = pd.concat([df_5, pd.DataFrame([row])], ignore_index=True)
        df_copy.drop(index, inplace=True)

# 单环
for index, row in df_copy.iterrows():
    smiles = row['smiles']
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        ring_info = mol.GetRingInfo()
        num_rings = ring_info.NumRings()
        if num_rings == 1:
            df_6 = pd.concat([df_6, pd.DataFrame([row])], ignore_index=True)
            df_copy.drop(index, inplace=True)

# 其他
df_7 = df_copy.copy()

In [ ]:
print(df_1.shape)
print(df_2.shape)
print(df_3.shape)
print(df_4.shape)
print(df_5.shape)
print(df_6.shape)
print(df_7.shape)

In [ ]:
dm_screening = dm_screening.reset_index(drop=True)

In [ ]:
processed_sam_gen = calculate_ecfp(dm_screening)

In [ ]:
processed_sam_gen

In [ ]:
# 获取 processed_sam_gen 的行数
num_samples = processed_sam_gen.shape[0]

# 创建 pvk_substrate_info 字典
pvk_substrate_info = {
    'Cs_ratio': 0.05,
    'FA_ratio': 0.9025,
    'Br_ratio': 0.15,
    'PVK_Eg': 1.56,
    'ITO': 0,
    'FTO': 0,
    'NiO': 1,
    'SnO2': 0,
    'ZnO': 0,
    'TiO2': 0
}

# 将 pvk_substrate_info 重复 num_samples 次，确保行数一致
df_predict_sample = pd.DataFrame([pvk_substrate_info] * num_samples)

df_predict_sample

In [ ]:
pvk_substrate_info_normalized = normalize_predict_sample(df_predict_sample, scaler_dict)

In [ ]:
pvk_substrate_info_normalized

In [ ]:
sam_gen_input = pd.concat([pvk_substrate_info_normalized, processed_sam_gen.iloc[:,0:]],axis=1)
sam_gen_input

In [ ]:
from sklearn.svm import SVC
# 设置最佳超参数组合
best_params = {
    'C': 50,
    'kernel': 'linear',
    'gamma': 0.0001
}
# 用于存储20次实验中每次新分子的预测结果
new_molecule_predictions_all = []

# 遍历重复实验
for i in tqdm(range(100), desc="Training Progress"):
    dfs = [df_1, df_2, df_3, df_4, df_5, df_6]
    df_test = pd.DataFrame()
    df_train = pd.DataFrame()

    for j, df_sub in enumerate(dfs, start=1):
        if len(df_sub) <= 10:
            test_indices = np.random.choice(df_sub.index, size=1, replace=False)
        elif 10 < len(df_sub) <= 20:
            test_indices = np.random.choice(df_sub.index, size=2, replace=False)
        elif 20 < len(df_sub) <= 30:
            test_indices = np.random.choice(df_sub.index, size=3, replace=False)
        elif 30 < len(df_sub) <= 40:
            test_indices = np.random.choice(df_sub.index, size=4, replace=False)
        elif 40 < len(df_sub) <= 50:
            test_indices = np.random.choice(df_sub.index, size=5, replace=False)
        elif 50 < len(df_sub):
            test_indices = np.random.choice(df_sub.index, size=6, replace=False)
        else:
            test_indices = []

        # 将测试集数据添加到 df_test，保持原始索引号
        df_test = pd.concat([df_test, df_sub.loc[test_indices]], ignore_index=False)

        # 将剩余的数据添加到 df_train，保持原始索引号
        train_indices = df_sub.index.difference(test_indices)
        df_train = pd.concat([df_train, df_sub.loc[train_indices]], ignore_index=False)

    # 分离特征和标签
    df_train_X = df_train.iloc[:, 1:-1].astype(float)
    df_test_X = df_test.iloc[:, 1:-1].astype(float)
    df_train_label = df_train['label'].astype(int)
    df_test_label = df_test['label'].astype(int)

    # 使用最优参数训练模型
    model = SVC(C=best_params['C'],
                kernel=best_params['kernel'],
                gamma=best_params['gamma'])
    model.fit(df_train_X, df_train_label)

    # 在测试集上进行预测
    predictions = model.predict(df_test_X)
    
    # 对新分子进行预测
    new_molecules_X = sam_gen_input.astype(float)  # 假设 sam_gen_input 包含新分子的特征数据
    new_molecule_predictions = model.predict(new_molecules_X)
    
    # 记录每次实验中对新分子的预测结果
    new_molecule_predictions_all.append(new_molecule_predictions)

# 将每次预测结果转换为 NumPy 数组，计算平均值和标准差
new_molecule_predictions_all = np.array(new_molecule_predictions_all)
mean_predictions = np.mean(new_molecule_predictions_all, axis=0)

# 创建 DataFrame 来记录平均值和标准差
df_predictions = pd.DataFrame({
    'Mean_Prediction': mean_predictions,
})

In [ ]:
df_predictions

In [ ]:
sam_gen_pred_efficiency = pd.concat([dm_screening.iloc[:,1:2], df_predictions],axis=1)
sam_gen_pred_efficiency

In [ ]:
sorted_pce_results = sam_gen_pred_efficiency.sort_values(by="Mean_Prediction", ascending=False)
sorted_pce_results

In [ ]:
top_10_pce_index = int(len(sorted_pce_results) * 0.1)

In [ ]:
pce_screening = sorted_pce_results.iloc[:top_10_pce_index]

In [ ]:
pce_screening

In [ ]:
pce_screening.shape

In [ ]:
pce_screening.iloc[:,0:1].to_csv("./pce_screening_0225.csv", index=False)

In [ ]:
def draw_molecule_with_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    img = Draw.MolToImage(mol, size=(300, 300))
    return img

In [ ]:
for index, row in pce_screening.iterrows():
    smiles = row['smiles']
    img = draw_molecule_with_smiles(smiles)
    
    display(img)
    print(f"smiles: {smiles}")

SAScore assessment

In [ ]:
sascore = pd.read_excel("./SAScore_for_sas_score_0225.xlsx")

In [ ]:
# 目标分子 SMILES 列表
target_smiles = [
    "N#C/C(=C\c1ccc(-c2ccc(N(c3ccccc3)c3ccccc3)cc2)cc1)P(=O)(O)O", 
    "N#CC(=Cc1ccc(-c2ccc(N(c3ccccc3)c3ccccc3)cc2)c2nsnc12)P(=O)(O)O",
    "N#CC(=Cc1ccc(-c2ccc(-c3ccc(N(c4ccccc4)c4ccccc4)cc3)c3nsnc23)cc1)P(=O)(O)O"
]

# 检查每个分子是否存在
for smiles in target_smiles:
    if smiles in sascore['SMILES'].values:
        print(f"{smiles} 存在于数据集中")
    else:
        print(f"{smiles} 不存在于数据集中")

In [ ]:
sascore

In [ ]:
def process_amount_column(column):
    # 如果是 None 或 'NaN'，替换为 ￥0
    column = column.fillna('￥0').replace('NaN', '￥0')
    # 统一去掉中文或英文的人民币符号
    column = column.str.replace('￥', '', regex=False).str.replace('¥', '', regex=False)
    # 转成 float
    column = column.astype(float)
    return column

In [ ]:
sascore['EstimatedCost'] = process_amount_column(sascore['EstimatedCost'])

In [ ]:
sascore

In [ ]:
SAScore_screening = sascore[(sascore['SAScore']<= 3)& (sascore['Confidence Score'] >= 0.6)]

In [ ]:
# 目标分子 SMILES 列表
target_smiles = [
    "N#C/C(=C\c1ccc(-c2ccc(N(c3ccccc3)c3ccccc3)cc2)cc1)P(=O)(O)O", 
    "N#CC(=Cc1ccc(-c2ccc(N(c3ccccc3)c3ccccc3)cc2)c2nsnc12)P(=O)(O)O",
    "N#CC(=Cc1ccc(-c2ccc(-c3ccc(N(c4ccccc4)c4ccccc4)cc3)c3nsnc23)cc1)P(=O)(O)O"
]

# 检查每个分子是否存在
for smiles in target_smiles:
    if smiles in SAScore_screening['SMILES'].values:
        print(f"{smiles} 存在于数据集中")
    else:
        print(f"{smiles} 不存在于数据集中")

In [ ]:
max(SAScore_screening['EstimatedCost'])

In [ ]:
cost_screening = SAScore_screening[(SAScore_screening['EstimatedCost']<= 20000)]

In [ ]:
# 目标分子 SMILES 列表
target_smiles = [
    "N#C/C(=C\c1ccc(-c2ccc(N(c3ccccc3)c3ccccc3)cc2)cc1)P(=O)(O)O", 
    "N#CC(=Cc1ccc(-c2ccc(N(c3ccccc3)c3ccccc3)cc2)c2nsnc12)P(=O)(O)O",
    "N#CC(=Cc1ccc(-c2ccc(-c3ccc(N(c4ccccc4)c4ccccc4)cc3)c3nsnc23)cc1)P(=O)(O)O"
]

# 检查每个分子是否存在
for smiles in target_smiles:
    if smiles in cost_screening['SMILES'].values:
        print(f"{smiles} 存在于数据集中")
    else:
        print(f"{smiles} 不存在于数据集中")

In [ ]:
step_screening = cost_screening[(cost_screening['Total Steps']<= 4)]

In [ ]:
# 目标分子 SMILES 列表
target_smiles = [
    "N#C/C(=C\c1ccc(-c2ccc(N(c3ccccc3)c3ccccc3)cc2)cc1)P(=O)(O)O", 
    "N#CC(=Cc1ccc(-c2ccc(N(c3ccccc3)c3ccccc3)cc2)c2nsnc12)P(=O)(O)O",
    "N#CC(=Cc1ccc(-c2ccc(-c3ccc(N(c4ccccc4)c4ccccc4)cc3)c3nsnc23)cc1)P(=O)(O)O"
]

# 检查每个分子是否存在
for smiles in target_smiles:
    if smiles in step_screening['SMILES'].values:
        print(f"{smiles} 存在于数据集中")
    else:
        print(f"{smiles} 不存在于数据集中")

In [ ]:
step_screening

In [ ]:
step_screening.iloc[:,1:2].to_csv("./sas_screening_0225.csv", index=False)

In [ ]:
def draw_molecule_with_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    img = Draw.MolToImage(mol, size=(300, 300))
    return img

In [ ]:
for index, row in step_screening.iterrows():
    smiles = row['SMILES']
    img = draw_molecule_with_smiles(smiles)
    
    display(img)
    print(f"smiles: {smiles}")